In [ ]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [7]:
# Check GPU status
!nvidia-smi

Thu Apr  2 15:25:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:83:00.0 Off |                  N/A |
|  0%   56C    P8             46W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# For Qwen2.5-VL
# # %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     %pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# %pip install transformers==4.56.2
# %pip install --no-deps trl==0.22.2

# For Qwen3-VL
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Configuration

In [9]:
SEED = 42

# Model configuration
MODEL_ID = 'alxxtexxr/Qwen3-VL-8B-Thinking-with-Codebook-8K'

# Data configuration
SFT_DATA_ID = 'alxxtexxr/coco_captions_1.25k_serialized_for_Qwen3-VL-8B-Thinking-with-Codebook-8K'

In [10]:
from datetime import datetime

# Resume training configuration
RESUME_MODEL_ID = None
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    # hub_model_id = RESUME_MODEL_ID
    # project_name = hub_model_id.split('/')[-1]
    # model_name = project_name

    # from huggingface_hub import snapshot_download
    # snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    raise NotImplementedError("Resuming from checkpoint is not implemented yet!")
else:
    # model_name = 'unsloth/Meta-Llama-3.1-8B'
    user_name, model_name = MODEL_ID.split('/')
    project_name = f'{model_name}-SFT-LoRA-v{datetime.now().strftime("%Y%m%d%H%M%S")}'
    hub_model_id = f'{user_name}/{project_name}'
print("Resume from checkpoint:", resume_from_checkpoint)
print("Project name:", project_name)
print("Hugging Face model ID:", hub_model_id)

Resume from checkpoint: False
Project name: Qwen3-VL-8B-Thinking-with-Codebook-8K-SFT-LoRA-v20260402152604
Hugging Face model ID: alxxtexxr/Qwen3-VL-8B-Thinking-with-Codebook-8K-SFT-LoRA-v20260402152604


In [11]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print(f"Random seed: {seed}")

set_seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Random seed: 42


In [12]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [13]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # use_gradient_checkpointing = False,
    # dtype = torch.float16, # Force FP16
)

==((====))==  Unsloth 2026.3.18: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.569 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/192 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [18]:
tokenizer.tokenizer.encode("<|vtok_7550|><|vtok_974|><|vtok_4167|><|vtok_4384|>")

[159219, 152643, 155836, 156053]

In [9]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True,  # False if not finetuning language layers
    finetune_attention_modules = True,  # False if not finetuning attention layers
    finetune_mlp_modules       = True,  # False if not finetuning MLP layers

    r = 16,          # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16, # Recommended alpha == r at least
    lora_dropout = 0,
    bias = 'none',
    random_state = SEED,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = 'all-linear', # Optional now! Can specify a list if needed
)

# Data

In [10]:
from datasets import load_dataset

dataset = load_dataset(SFT_DATA_ID)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
        num_rows: 1000
    })
    val: Dataset({
        features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
        num_rows: 125
    })
    test: Dataset({
        features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
        num_rows: 125
    })
})


In [11]:
train_set = dataset['train']
val_set = dataset['val']

print("Train set:")
print(train_set)
print("\nValidation set:")
print(val_set)

Train set:
Dataset({
    features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
    num_rows: 125
})


In [12]:
def convert_to_conversation(sample):
    conversation = [
        { 
            "role": "user",
            "content" : [
                {"type" : "text",  "text"  : sample['caption']},
                # {"type" : "image", "image" : sample["image"]},
            ],
        },
        { 
            "role" : "assistant",
            "content" : [
                {"type" : "text",  "text"  : sample["image_serialized"]},
            ],
        },
    ]
    return { "messages" : conversation }

train_dataset = [convert_to_conversation(sample) for sample in train_set]
val_dataset = [convert_to_conversation(sample) for sample in val_set]

print("Train dataset sample:")
print(train_dataset[0])
print("\nValidation dataset sample:")
print(val_dataset[0])

Train dataset sample:
{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'A man wearing a glove with a bird on top of it.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '<|vtok_2012|><|vtok_1621|><|vtok_3723|><|vtok_2518|><|vtok_2518|><|vtok_2518|><|vtok_2428|><|vtok_6713|><|vtok_2518|><|vtok_5156|><|vtok_5371|><|vtok_1092|><|vtok_154|><|vtok_7997|><|vtok_5371|><|vtok_3535|><|vtok_6980|><|vtok_281|><|vtok_3765|><|vtok_7461|><|vtok_2518|><|vtok_1092|><|vtok_6713|><|vtok_6713|><|vtok_181|><|vtok_3083|><|vtok_4122|><|vtok_4027|><|vtok_3179|><|vtok_3083|><|vtok_3437|><|vtok_6193|><|vtok_2681|><|vtok_846|><|vtok_262|><|vtok_2518|><|vtok_1092|><|vtok_7959|><|vtok_185|><|vtok_2428|><|vtok_6713|><|vtok_6713|><|vtok_2518|><|vtok_4122|><|vtok_7792|><|vtok_7926|><|vtok_3884|><|vtok_4047|><|vtok_7817|><|vtok_5278|><|vtok_2518|><|vtok_7382|><|vtok_2428|><|vtok_2428|><|vtok_5733|><|vtok_3186|><|vtok_6713|><|vtok_6713|><|vtok_6713|><|vtok_3700|><|vtok_7366|><|vtok_13

In [13]:
FastVisionModel.for_inference(model) # Enable for inference!

image = None
instruction = "Several slices of bread with side salads on a table."

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

<|im_start|>user
Several slices of bread with side salads on a table.<|im_end|>
<|im_start|>assistant
Here’s a vivid description of the scene:

**Several slices of bread with side salads on a table** — the golden-brown crusts of freshly baked bread slices, slightly toasted and glistening with olive oil, rest invitingly in a rustic ceramic bowl or scattered casually on a weathered wooden board. Each slice is generously layered with vibrant, colorful salads — crisp cucumber ribbons, juicy cherry tomatoes halved, tangy feta crumbles, and a scattering of fresh basil leaves. A drizzle of bright herb vinaigrette glistens in the soft afternoon light, catching the edge of each leaf. A small terracotta dish holds a mound of creamy hummus beside the bread, while a wedge of lemon and a handful of olives sit nearby, ready to add a burst of flavor. A fork lies forgotten beside the bowl, hinting at the midday meal that’s just been enjoyed — or is about to begin. The air is warm and fragrant with toa

# Training

In [14]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        # Training arguments
        seed = SEED,
        
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        
        # max_steps = 50, # For testing
        num_train_epochs = 3, # Set this instead of max_steps for full training runs
        warmup_ratio=0.05,
        # warmup_steps = 5,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        max_grad_norm=1.0,
        weight_decay = 0.01,
        
        # Validation arguments
        eval_strategy='steps',
        eval_steps=10,
        
        # Logging arguments
        logging_strategy='steps',
        logging_steps=10,
        # logging_first_step=True,
        report_to=['tensorboard', 'wandb'],
        
        # Saving arguments
        save_strategy='steps',
        # save_steps=1, # For testing
        save_steps=20,
        # save_total_limit=5, # 1 best + 4 recent checkpoints. Warning: It doesn't work
        
        # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
        # So you will find one checkpoint at the end of each epoch.
        # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
        # load_best_model_at_end=True, 

        output_dir=project_name,
        hub_model_id=hub_model_id,
        push_to_hub=True,

        hub_strategy='all_checkpoints',
        hub_always_push=True,

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 1024,
    ),
)

Unsloth: Model does not have a default image size - using 512


In [15]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 3 | Total steps = 96
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 43,646,976 of 8,875,692,272 (0.49% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,10.082700,8.234425
20,7.618500,7.254294
30,6.816100,6.721241
40,6.311800,6.361493
50,5.981500,6.114259
60,5.760200,5.958696
70,5.598000,5.874349
80,5.492800,5.825309
90,5.477000,5.813007


# Inference

In [16]:
FastVisionModel.for_inference(model) # Enable for inference!

image = None
instruction = "Several slices of bread with side salads on a table."

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

In [17]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

<|im_start|>user
Several slices of bread with side salads on a table.<|im_end|>
<|im_start|>assistant
<|vtok_5424|><|vtok_1834|><|vtok_5261|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><